Analyse du lien entre statut et gravité, sur la durée des séjours

In [18]:
import pandas as pd

#importation des données sous forme de dataframes
MCO2022 = pd.read_csv("SAE/2022/MCO_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
SSR2022 = pd.read_csv("SAE/2022/SSR_2022r.csv", sep=";", encoding="latin-1")

UrgP2022 = pd.read_csv("SAE/2022/URGENCES_P_2022a.csv", sep=";",encoding="latin-1")

FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

#ajout d'une colonne aux données indiuant le statut de chaque établissement
MCO2022 = MCO2022.merge(FINESS,on='FI',how='left')
Urg2022 = Urg2022.merge(FINESS, on='FI', how='left')
SSR2022 = SSR2022.merge(FINESS, on='FI', how='left')

In [19]:
MCO2022.head(3)

,BOR,AN,FI,RS,FI_EJ,LIT_MED,JLI_MED,SEJHC_MED,SEJ0_MED,JOU_MED,...,DNEU,DPED,DOPH,ACTCLI_PM,ACTCLI_SAG,ACTTEC_PM,ACTTEC_DEN,ACTTEC_SAG,ACTTEC_PNM,Statut
0,MCO,2022,010000024,CH DE FLEYRIAT,010780054,270.0,92419.0,14972.0,4042.0,82796.0,...,105.0,105.0,NaN,92931.0,2523.0,63629.0,1204.0,5516.0,12582.0,Public
1,MCO,2022,010000032,CH BUGEY SUD,010780062,66.0,24090.0,4020.0,771.0,21185.0,...,90.0,24.0,NaN,17899.0,1822.0,21383.0,0.0,1676.0,7021.0,Public
2,MCO,2022,010000065,CH DE TREVOUX - MONTPENSIER,010780096,59.0,21535.0,1669.0,9.0,18131.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Public


In [20]:
hospidiag = pd.read_csv("Hospidiag/hd2022.csv", sep=";", encoding='latin-1')
hospidiag.head(3)

,finess,rs,champ_pmsi,taa,cat,taille_MCO,taille_M,taille_C,taille_O,A7,...,RH1,RH2,RH3,RH4,RH5,RH6,RH7,RH8,RH9,RH10
0,010007300,CLINIQUE AMBULATOIRE CENDANEG,OQN,TAA,CLI,T1,M1,C1,NaN,"15,2",...,NaN,"31004,3",53812,NaN,"0,3",NaN,NaN,.z,.z,.z
1,010007987,CH HAUTEVILLE,DGF,TAA,CH,T1,M1,C0,NaN,"27,7",...,NaN,NaN,NaN,"15,1",NaN,NaN,NaN,"7,5","42,8",NaN
2,010008407,CH DU HAUT BUGEY,DGF,TAA,CH,T2,M2,C2,O2,"7,2",...,29.0,"14727,9","54779,5","39,8","1,4","5,5",NaN,NaN,NaN,NaN


In [21]:
# Normalisation du finess 
MCO2022['FI'] = MCO2022['FI'].astype(str).str.zfill(9)
MCO2022['FI_EJ'] = MCO2022['FI_EJ'].astype(str).str.zfill(9)
hospidiag['finess'] = hospidiag['finess'].astype(str).str.zfill(9)

In [22]:
import numpy as np

MCO2022['Statut'] = MCO2022['Statut'].astype(str).str.strip()
condition_public = MCO2022['Statut'] == 'Public'

MCO2022['FI_Hospidiag'] = np.where(
    condition_public, 
    MCO2022['FI_EJ'],  # Valeur si Vrai (Public)
    MCO2022['FI']      # Valeur si Faux (Privé)
)

# On s'assure que cette nouvelle colonne est bien une chaîne de 9 caractères
MCO2022['FI_Hospidiag'] = MCO2022['FI_Hospidiag'].astype(str).str.split('.').str[0].str.zfill(9)

In [23]:
# Distinction CHU et Public
chu = pd.read_excel("CHRU.xlsx")
chu["FINESS"] = chu["FINESS"].astype(object)
chu = chu.rename(columns={"FINESS":"FI"})
chu['FI'] = chu['FI'].apply(lambda x: '0' + str(x) if len(str(x)) == 8 else str(x))
MCO2022.loc[MCO2022['FI'].isin(chu['FI']), "Statut"] = "CHU"  #Ajout du statut CHU dans la colonne Statut de FINESS

In [24]:
df_hosp_clean = hospidiag[['finess', 'A9']]

df_final = pd.merge(
    MCO2022, 
    df_hosp_clean, 
    left_on='FI_Hospidiag', 
    right_on='finess', 
    how='left'
)

df_final = df_final.drop(columns=['finess'])

In [32]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Copie et nettoyage initial
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()

# 2. Nettoyage CRITIQUE du Statut AVANT la conversion en catégorie
# On convertit en string, on nettoie les espaces, et on remplace les chaines "nan" par de vrais NaN
df_clean['Statut'] = df_clean['Statut'].astype(str).str.strip()
df_clean['Statut'] = df_clean['Statut'].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})

# 3. Suppression des lignes vides (Statut OU A9)
df_clean = df_clean.dropna(subset=['Statut', 'A9'])

# 4. Conversion A9 et Calcul DMS
df_clean['A9'] = pd.to_numeric(df_clean['A9'].astype(str).str.replace(',', '.'))
df_clean['DMS'] = df_clean['JOU_MCO'] / df_clean['SEJHC_MCO']
df_clean['log_JOU_MCO'] = np.log1p(df_clean['JOU_MCO'])
df_clean['log_DMS'] = np.log1p(df_clean['DMS'])
df_clean['log_SEJHC_MCO'] = np.log1p(df_clean['SEJHC_MCO'])

# 5. Définir le Statut avec une référence explicite (ex: "Public")
# Cela permet de comparer tout le monde par rapport au Public
# "C(Statut, Treatment(reference='Public'))" dit à Python : "Public est mon zéro".
model = smf.ols("log_JOU_MCO ~ C(Statut, Treatment(reference='Public')) * A9 + log_SEJHC_MCO", data=df_clean).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            log_JOU_MCO   R-squared:                       0.914
Model:                            OLS   Adj. R-squared:                  0.914
Method:                 Least Squares   F-statistic:                     1840.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:06:20   Log-Likelihood:                -794.25
No. Observations:                1393   AIC:                             1607.
Df Residuals:                    1384   BIC:                             1654.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                                                        coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [26]:
df_clean['A9'].describe()

count    1393.000000
mean       15.506095
std        16.495951
min         0.000000
25%         3.690000
50%        11.170000
75%        16.750000
max        94.740000
Name: A9, dtype: float64

# Ajout pour le rapport final

## Table 9 — Sans A9 (version de base pour comparaison)

In [27]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --- Préparation identique ---
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()
df_clean['Statut'] = df_clean['Statut'].astype(str).str.strip()
df_clean['Statut'] = df_clean['Statut'].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})
df_clean = df_clean.dropna(subset=['Statut', 'A9'])
df_clean['A9'] = pd.to_numeric(df_clean['A9'].astype(str).str.replace(',', '.'))
df_clean['DMS'] = df_clean['JOU_MCO'] / df_clean['SEJHC_MCO']
df_clean['log_JOU_MCO']    = np.log1p(df_clean['JOU_MCO'])
df_clean['log_SEJHC_MCO']  = np.log1p(df_clean['SEJHC_MCO'])

# --- Modèle SANS A9 ---
model_sans_A9 = smf.ols(
    "log_JOU_MCO ~ C(Statut, Treatment(reference='Public')) + log_SEJHC_MCO",
    data=df_clean
).fit()

print("=== TABLE 9 — Sans A9 ===")
print(model_sans_A9.summary())

=== TABLE 9 — Sans A9 ===
                            OLS Regression Results                            
Dep. Variable:            log_JOU_MCO   R-squared:                       0.900
Model:                            OLS   Adj. R-squared:                  0.900
Method:                 Least Squares   F-statistic:                     3130.
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        22:28:57   Log-Likelihood:                -898.36
No. Observations:                1393   AIC:                             1807.
Df Residuals:                    1388   BIC:                             1833.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------

## Table 10 — Variable dépendante = DMS (log)

Le coefficient "égal à 1" vient du fait que log_DMS = log_JOU - log_SEJHC : utiliser la DMS comme dépendante revient implicitement à contraindre le coefficient de log_SEJHC_MCO à 1 dans le modèle précédent. Ici ce coefficient n'apparaît donc plus librement — c'est l'identité comptable.

In [28]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# --- Préparation ---
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()
df_clean['Statut'] = df_clean['Statut'].astype(str).str.strip()
df_clean['Statut'] = df_clean['Statut'].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})
df_clean = df_clean.dropna(subset=['Statut', 'A9'])
df_clean['A9'] = pd.to_numeric(df_clean['A9'].astype(str).str.replace(',', '.'))
df_clean['DMS'] = df_clean['JOU_MCO'] / df_clean['SEJHC_MCO']
df_clean['log_DMS'] = np.log1p(df_clean['DMS'])

# --- Modèle avec log_DMS comme dépendante ---
# On n'inclut PAS log_SEJHC_MCO car il est "absorbé" dans la définition de la DMS
# (log_DMS = log_JOU - log_SEJHC => contrainte implicite coef=1 sur log_SEJHC)
model_dms = smf.ols(
    "log_DMS ~ C(Statut, Treatment(reference='Public')) * A9",
    data=df_clean
).fit()

print("=== TABLE 10 — Variable dépendante : log(DMS) ===")
print("NB : utiliser log_DMS revient à contraindre le coef. de log_SEJHC_MCO à 1")
print("     dans le modèle sur log_JOU_MCO (identité comptable DMS = JOU/SEJ)")
print(model_dms.summary())

# --- Vérification de la contrainte ---
# Si on réintroduit log_SEJHC_MCO, son coeff devrait être ~0 (déjà 'retiré' par la DMS)
model_dms_check = smf.ols(
    "log_DMS ~ C(Statut, Treatment(reference='Public')) * A9 + log_SEJHC_MCO",
    data=df_clean.assign(log_SEJHC_MCO=np.log1p(df_clean['SEJHC_MCO']))
).fit()
print("\n--- Vérification : coef. log_SEJHC_MCO dans le modèle DMS ---")
print(f"Coef log_SEJHC_MCO = {model_dms_check.params['log_SEJHC_MCO']:.4f}  (attendu ≈ 0)")

=== TABLE 10 — Variable dépendante : log(DMS) ===
NB : utiliser log_DMS revient à contraindre le coef. de log_SEJHC_MCO à 1
     dans le modèle sur log_JOU_MCO (identité comptable DMS = JOU/SEJ)
                            OLS Regression Results                            
Dep. Variable:                log_DMS   R-squared:                       0.495
Model:                            OLS   Adj. R-squared:                  0.493
Method:                 Least Squares   F-statistic:                     194.0
Date:                Sun, 26 Apr 2026   Prob (F-statistic):          1.88e-200
Time:                        22:28:57   Log-Likelihood:                -699.76
No. Observations:                1393   AIC:                             1416.
Df Residuals:                    1385   BIC:                             1457.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
               

## Table 11 — IV/2SLS : instrumentation par les données 2019

In [29]:
# ================================================================
# CONSTRUCTION DE df_2019 — instrument pour la Table 11
# À placer EN AMONT du code Table 11 dans le notebook
# ================================================================

# 1. Chargement des fichiers SAE 2019 (même structure que 2022)
MCO2019 = pd.read_csv("SAE/2019/MCO_2019.csv", sep=";", encoding="latin-1")

# 2. Normalisation du FINESS (identique à 2022)
MCO2019['FI']    = MCO2019['FI'].astype(str).str.zfill(9)
MCO2019['FI_EJ'] = MCO2019['FI_EJ'].astype(str).str.zfill(9)

# 3. Merge avec FINESS pour récupérer le Statut
MCO2019 = MCO2019.merge(FINESS, on='FI', how='left')  # FINESS déjà chargé plus haut

# 4. Même logique Public → FI_EJ, Privé → FI pour matcher Hospidiag
MCO2019['Statut'] = MCO2019['Statut'].astype(str).str.strip()
MCO2019['FI_Hospidiag'] = np.where(
    MCO2019['Statut'] == 'Public',
    MCO2019['FI_EJ'],
    MCO2019['FI']
)
MCO2019['FI_Hospidiag'] = (MCO2019['FI_Hospidiag']
                            .astype(str)
                            .str.split('.')
                            .str[0]
                            .str.zfill(9))

# 5. On ne garde que les colonnes utiles pour l'instrument
#    FI_Hospidiag sera la clé de jointure, SEJHC_MCO est l'instrument
df_2019 = (MCO2019[['FI_Hospidiag', 'SEJHC_MCO']]
           .rename(columns={
               'FI_Hospidiag': 'finess',
               'SEJHC_MCO':    'SEJHC_MCO_2019'
           })
           .dropna(subset=['SEJHC_MCO_2019'])
           .query('SEJHC_MCO_2019 > 0'))

# 6. Agrégation : si plusieurs lignes par finess (établissements multi-lignes),
#    on somme les séjours comme pour les données 2022
df_2019 = (df_2019
           .groupby('finess', as_index=False)['SEJHC_MCO_2019']
           .sum())

print(f"df_2019 : {len(df_2019)} établissements")
print(df_2019.head())

df_2019 : 1251 établissements
      finess  SEJHC_MCO_2019
0  010007987             678
1  010008407            6649
2  010009132             337
3  010780054           21660
4  010780062            5526


In [30]:
import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS  # pip install linearmodels

# --- Préparation ---
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()
df_clean['Statut'] = df_clean['Statut'].astype(str).str.strip()
df_clean['Statut'] = df_clean['Statut'].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})
df_clean = df_clean.dropna(subset=['Statut', 'A9'])
df_clean['A9'] = pd.to_numeric(df_clean['A9'].astype(str).str.replace(',', '.'))
df_clean['log_JOU_MCO']   = np.log1p(df_clean['JOU_MCO'])
df_clean['log_SEJHC_MCO'] = np.log1p(df_clean['SEJHC_MCO'])

# --- Merge avec les données 2019 (instrument) ---
# Hypothèse : tu as un df_2019 avec colonnes ['finess', 'SEJHC_MCO_2019']
# (remplace 'finess' par ta clé de jointure réelle)
df_iv = df_clean.merge(
    df_2019,                    # colonnes : ['finess', 'SEJHC_MCO_2019']
    left_on='FI_Hospidiag',     # ← clé dans df_clean
    right_on='finess',          # ← clé dans df_2019
    how='inner'
)
df_iv['log_SEJHC_2019'] = np.log1p(df_iv['SEJHC_MCO_2019'])
df_iv = df_iv.dropna(subset=['log_SEJHC_2019'])

# --- Encodage des dummies Statut (linearmodels n'utilise pas de formule Patsy) ---
df_iv = pd.get_dummies(df_iv, columns=['Statut'], drop_first=False)
df_iv.columns = (df_iv.columns
                 .str.replace(' ', '_', regex=False)
                 .str.replace('é', 'e', regex=False)
                 .str.replace('û', 'u', regex=False)
                 .str.replace('ê', 'e', regex=False)
                 .str.replace('â', 'a', regex=False))

statut_cols = [c for c in df_iv.columns if c.startswith('Statut_') and 'Public' not in c]
print("Colonnes Statut utilisées :", statut_cols)
# Attendu : ['Statut_CHU', 'Statut_Prive_lucratif', 'Statut_Prive_non_lucratif']

# --- Variables du modèle ---
endog  = df_iv['log_JOU_MCO']                            # Y
exog   = df_iv[statut_cols + ['A9']].assign(const=1)     # X exogènes + constante
instru = df_iv['log_SEJHC_MCO']                          # variable endogène
instr  = df_iv[['log_SEJHC_2019'] + statut_cols + ['A9']].assign(const=1)  # instruments

# --- Estimation 2SLS ---
model_iv = IV2SLS(
    dependent=endog,
    exog=exog,
    endog=instru,          # log_SEJHC_MCO est endogène
    instruments=df_iv[['log_SEJHC_2019']]  # instrumenté par la valeur 2019
).fit(cov_type='robust')

print("=== TABLE 11 — IV/2SLS : log_SEJHC_MCO instrumenté par log_SEJHC_2019 ===")
print(model_iv.summary)

# --- Test de la pertinence de l'instrument (1ère étape) ---
from linearmodels.iv.results import compare
from statsmodels.formula.api import ols as sm_ols

first_stage = sm_ols(
    "log_SEJHC_MCO ~ log_SEJHC_2019 + " + " + ".join(statut_cols) + " + A9",
    data=df_iv
).fit()

print("\n--- Première étape (pertinence de l'instrument) ---")
print(f"F-stat sur l'instrument : {first_stage.fvalue:.2f}  (seuil usuel > 10)")
print(f"Coef log_SEJHC_2019     : {first_stage.params['log_SEJHC_2019']:.4f}")
print(f"R² 1ère étape           : {first_stage.rsquared:.4f}")

# --- Test de Hausman (endogénéité) ---
# Si les coefs IV et OLS diffèrent significativement => endogénéité confirmée
model_ols = sm_ols(
    "log_JOU_MCO ~ log_SEJHC_MCO + " + " + ".join(statut_cols) + " + A9",
    data=df_iv
).fit()

print("\n--- Comparaison OLS vs IV (test informel de Hausman) ---")
print(f"Coef log_SEJHC_MCO (OLS) : {model_ols.params['log_SEJHC_MCO']:.4f}")
print(f"Coef log_SEJHC_MCO (IV)  : {model_iv.params['log_SEJHC_MCO']:.4f}")

Colonnes Statut utilisées : ['Statut_CHU', 'Statut_Prive_lucratif', 'Statut_Prive_non_lucratif']
=== TABLE 11 — IV/2SLS : log_SEJHC_MCO instrumenté par log_SEJHC_2019 ===
                          IV-2SLS Estimation Summary                          
Dep. Variable:            log_JOU_MCO   R-squared:                      0.9108
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9105
No. Observations:                1359   F-statistic:                    7197.5
Date:                Sun, Apr 26 2026   P-value (F-stat)                0.0000
Time:                        22:28:58   Distribution:                  chi2(5)
Cov. Estimator:                robust                                         
                                                                              
                                     Parameter Estimates                                     
                           Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------